<p align="center">
  <img src="https://images.unsplash.com/photo-1555949963-ff9fe0c870eb?auto=format&fit=crop&w=1600&q=85"
       width="100%" alt="Web Intelligence and Natural Language Processing">
</p>

<h1 align="center">🌐 Web Intelligence and NLP-Based Sentiment Analytics</h1>

<p align="center">
  <b>Notebook 01 — Web Scraping</b><br>
  End-to-End Web Data Collection Pipeline
</p>

---

## 📌 Project Overview

This project builds an end-to-end **Web Intelligence and NLP Analytics** pipeline that collects publicly available textual data, converts unstructured web content into structured data, and prepares it for downstream Natural Language Processing (NLP) and sentiment analysis.

This notebook represents the **first stage: Web Scraping**.

### 🎯 Objective

The objective of this notebook is to:

- Identify a suitable public web data source
- Send HTTP requests safely
- Retrieve and parse HTML content
- Inspect the page structure
- Extract relevant textual information
- Handle multiple pages through pagination
- Build a structured Pandas DataFrame
- Perform basic raw-data validation
- Save the collected dataset for the next notebook

### 🔄 End-to-End Project

```text
🌐 Web Source
      ↓
🕷️ Web Scraping
      ↓
📦 Raw Dataset
      ↓
🧹 Data Cleaning & EDA
      ↓
📝 NLP Preprocessing
      ↓
🤖 Sentiment Analysis
      ↓
🔑 Keyword & Topic Analysis
      ↓
📊 Final Insights
      ↓
🚀 Interactive Dashboard
```

## 📓 Notebook Workflow

```text
01. Project Overview
02. Import Libraries
03. Environment & Configuration
04. Define Target Web Source
05. Send HTTP Request
06. Validate Response
07. Parse HTML
08. Inspect Web Page Structure
09. Extract Required Fields
10. Handle Pagination
11. Create DataFrame
12. Data Validation
13. Save Raw Dataset
14. Final Summary
```

## 🌐 Data Source

### Quotes to Scrape

For this project, we will initially use **Quotes to Scrape**, a public practice website specifically designed for learning and testing web-scraping techniques.

**Website:** `https://quotes.toscrape.com/`

The website provides structured textual content such as:

- Quote text
- Author name
- Tags
- Source page

This makes it suitable for demonstrating the complete scraping pipeline before applying NLP and sentiment analysis.

> **Data Collection Note:** Always use publicly accessible sources where automated collection is permitted. Review the source website's terms, robots.txt, rate limits, and applicable restrictions before collecting data. Do not collect private, login-protected, or sensitive personal information.

## 1. Import Libraries

Import the core Python libraries required for HTTP requests, HTML parsing, data manipulation, file handling, and request timing.

In [1]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

# Data Manipulation
import pandas as pd
import numpy as np

# Web Scraping
import requests
from bs4 import BeautifulSoup

# File & Path Management
from pathlib import Path

# Request Timing
import time

# Warning Control
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Environment & Configuration

Configure project directories and basic scraping parameters.

Keeping paths, headers, timeout, and delay settings centralized makes the notebook easier to maintain and reproduce.

In [2]:
# ============================================================
# ENVIRONMENT & CONFIGURATION
# ============================================================

# Project root
PROJECT_ROOT = Path.cwd().parent

# Data directories
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
FINAL_DATA_DIR = DATA_DIR / "final"

# Create directories
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Scraping configuration
REQUEST_TIMEOUT = 30
REQUEST_DELAY = 1

# HTTP headers
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
}

print("Environment configured successfully.")
print(f"Project Root : {PROJECT_ROOT}")
print(f"Raw Data Dir : {RAW_DATA_DIR}")

Environment configured successfully.
Project Root : D:\Data-Science-Project\Data & Resources\Web-Intelligence-NLP
Raw Data Dir : D:\Data-Science-Project\Data & Resources\Web-Intelligence-NLP\data\raw


## 3. Define Target Web Source

Define the starting URL and scraping parameters.

The scraper will begin from the first page and follow the site's pagination until no further page is available.

In [3]:
# ============================================================
# TARGET WEB SOURCE
# ============================================================

BASE_URL = "https://quotes.toscrape.com/"
START_URL = BASE_URL

MAX_PAGES = None  # None = scrape all available pages

print(f"Starting URL : {START_URL}")
print(f"Maximum Pages: {MAX_PAGES if MAX_PAGES else 'All'}")

Starting URL : https://quotes.toscrape.com/
Maximum Pages: All


## 4. Send HTTP Request

Send an HTTP GET request to the target web page.

A timeout is included so that the request does not remain open indefinitely.

In [4]:
# ============================================================
# SEND HTTP REQUEST
# ============================================================

response = requests.get(
    START_URL,
    headers=HEADERS,
    timeout=REQUEST_TIMEOUT
)

print(f"Status Code : {response.status_code}")
print(f"Page Length : {len(response.text):,} characters")

Status Code : 200
Page Length : 11,021 characters


## 5. Validate Response

Validate the HTTP response before parsing the page.

A successful `200` status code indicates that the server returned the requested page successfully.

In [5]:
# ============================================================
# VALIDATE RESPONSE
# ============================================================

response.raise_for_status()

if response.status_code == 200:
    print("✓ Request successful.")
else:
    print(f"⚠ Unexpected status code: {response.status_code}")

✓ Request successful.


## 6. Parse HTML

Convert the HTML response into a searchable BeautifulSoup object.

This allows us to navigate the page structure and locate the elements containing quotes, authors, tags, and pagination links.

In [6]:
# ============================================================
# PARSE HTML
# ============================================================

soup = BeautifulSoup(response.text, "html.parser")

print(f"Page Title: {soup.title.get_text(strip=True)}")
print("HTML parsed successfully.")

Page Title: Quotes to Scrape
HTML parsed successfully.


## 7. Inspect Web Page Structure

Before extracting data, inspect the HTML structure.

The page stores each quote inside a `div` element with the CSS class `quote`.

Each quote contains:

- Quote text
- Author
- Tags

The pagination area contains the link to the next page.

In [7]:
# ============================================================
# INSPECT PAGE STRUCTURE
# ============================================================

quote_blocks = soup.select("div.quote")

print(f"Quote blocks found: {len(quote_blocks)}")

if quote_blocks:
    first_quote = quote_blocks[0]
    print("\nFirst quote preview:\n")
    print(first_quote.get_text(" ", strip=True))

Quote blocks found: 10

First quote preview:

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.” by Albert Einstein (about) Tags: change deep-thoughts thinking world


## 8. Extract Required Fields

Extract the fields required for the project.

### Target Schema

```text
quote_id
quote_text
author
tags
source_url
```

The extracted records will be stored as dictionaries and later converted into a Pandas DataFrame.

In [8]:
# ============================================================
# EXTRACT DATA FROM ONE PAGE
# ============================================================

def extract_quotes_from_page(soup, page_url):
    # Extract quote records from a parsed Quotes to Scrape page.

    records = []

    for quote in soup.select("div.quote"):
        text_element = quote.select_one("span.text")
        author_element = quote.select_one("small.author")

        tags = [
            tag.get_text(strip=True)
            for tag in quote.select("div.tags a.tag")
        ]

        record = {
            "quote_text": text_element.get_text(strip=True) if text_element else None,
            "author": author_element.get_text(strip=True) if author_element else None,
            "tags": ", ".join(tags),
            "source_url": page_url
        }

        records.append(record)

    return records


page_records = extract_quotes_from_page(soup, START_URL)

print(f"Records extracted from first page: {len(page_records)}")
print("\nSample record:")
print(page_records[0] if page_records else "No records found.")

Records extracted from first page: 10

Sample record:
{'quote_text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'author': 'Albert Einstein', 'tags': 'change, deep-thoughts, thinking, world', 'source_url': 'https://quotes.toscrape.com/'}


## 9. Handle Pagination

The website contains multiple pages of quotes.

The scraper will:

1. Process the current page
2. Extract all available records
3. Identify the `Next` page
4. Move to the next URL
5. Repeat until no next page exists

A small delay is used between requests to avoid sending requests too quickly.

In [9]:
# ============================================================
# PAGINATION & COMPLETE SCRAPING
# ============================================================

def scrape_quotes(start_url, max_pages=None, delay=1):
    # Scrape quotes from all available pages.

    all_records = []
    current_url = start_url
    page_number = 1

    while current_url:
        print(f"Scraping page {page_number}: {current_url}")

        response = requests.get(
            current_url,
            headers=HEADERS,
            timeout=REQUEST_TIMEOUT
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        page_records = extract_quotes_from_page(
            soup=soup,
            page_url=current_url
        )

        all_records.extend(page_records)

        if max_pages is not None and page_number >= max_pages:
            break

        next_link = soup.select_one("li.next a")

        if next_link and next_link.get("href"):
            current_url = BASE_URL.rstrip("/") + next_link["href"]
            page_number += 1
            time.sleep(delay)
        else:
            current_url = None

    return all_records


records = scrape_quotes(
    start_url=START_URL,
    max_pages=MAX_PAGES,
    delay=REQUEST_DELAY
)

print("\n" + "=" * 60)
print(f"Total records collected: {len(records):,}")
print("=" * 60)

Scraping page 1: https://quotes.toscrape.com/
Scraping page 2: https://quotes.toscrape.com/page/2/
Scraping page 3: https://quotes.toscrape.com/page/3/
Scraping page 4: https://quotes.toscrape.com/page/4/
Scraping page 5: https://quotes.toscrape.com/page/5/
Scraping page 6: https://quotes.toscrape.com/page/6/
Scraping page 7: https://quotes.toscrape.com/page/7/
Scraping page 8: https://quotes.toscrape.com/page/8/
Scraping page 9: https://quotes.toscrape.com/page/9/
Scraping page 10: https://quotes.toscrape.com/page/10/

Total records collected: 100


## 10. Create DataFrame

Convert the extracted records into a Pandas DataFrame.

A DataFrame provides a structured representation of the scraped information and makes it easier to validate and store the raw dataset.

In [10]:
# ============================================================
# CREATE DATAFRAME
# ============================================================

raw_df = pd.DataFrame(records)

raw_df.insert(
    0,
    "quote_id",
    range(1, len(raw_df) + 1)
)

print(f"Dataset Shape: {raw_df.shape}")
display(raw_df.head())

Dataset Shape: (100, 5)


,quote_id,quote_text,author,tags,source_url
0,1,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world",https://quotes.toscrape.com/
1,2,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices",https://quotes.toscrape.com/
2,3,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles",https://quotes.toscrape.com/
3,4,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor",https://quotes.toscrape.com/
4,5,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational",https://quotes.toscrape.com/


## 11. Data Validation

Perform basic validation on the raw dataset.

The validation checks:

- Dataset dimensions
- Column names
- Data types
- Missing values
- Duplicate records
- Text samples

This ensures that the scraping process generated a usable dataset before it is saved.

In [11]:
# ============================================================
# DATA VALIDATION
# ============================================================

print("Dataset Shape:")
print(raw_df.shape)

print("\nColumns:")
print(raw_df.columns.tolist())

print("\nData Types:")
display(raw_df.dtypes.to_frame("dtype"))

print("\nMissing Values:")
display(raw_df.isna().sum().to_frame("missing_count"))

print("\nDuplicate Rows:")
print(raw_df.duplicated().sum())

print("\nDuplicate Quote Text:")
print(raw_df["quote_text"].duplicated().sum())

print("\nText Length Summary:")
display(raw_df["quote_text"].str.len().describe())

Dataset Shape:
(100, 5)

Columns:
['quote_id', 'quote_text', 'author', 'tags', 'source_url']

Data Types:


,dtype
quote_id,int64
quote_text,str
author,str
tags,str
source_url,str



Missing Values:


,missing_count
quote_id,0
quote_text,0
author,0
tags,0
source_url,0



Duplicate Rows:
0

Duplicate Quote Text:
0

Text Length Summary:


count     100.000000
mean      122.270000
std       133.747376
min        34.000000
25%        66.500000
50%        86.000000
75%       125.000000
max      1084.000000
Name: quote_text, dtype: float64

## 12. Inspect Final Raw Data

Review a representative sample of the collected records before saving the dataset.

The raw dataset should preserve the information as it was extracted from the source, with only structural changes such as assigning an ID.

In [12]:
# ============================================================
# FINAL RAW DATA PREVIEW
# ============================================================

display(
    raw_df[
        ["quote_id", "quote_text", "author", "tags", "source_url"]
    ].head(10)
)

,quote_id,quote_text,author,tags,source_url
0,1,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world",https://quotes.toscrape.com/
1,2,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices",https://quotes.toscrape.com/
2,3,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles",https://quotes.toscrape.com/
3,4,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor",https://quotes.toscrape.com/
4,5,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational",https://quotes.toscrape.com/
5,6,“Try not to become a man of success. Rather be...,Albert Einstein,"adulthood, success, value",https://quotes.toscrape.com/
6,7,“It is better to be hated for what you are tha...,André Gide,"life, love",https://quotes.toscrape.com/
7,8,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,"edison, failure, inspirational, paraphrased",https://quotes.toscrape.com/
8,9,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,misattributed-eleanor-roosevelt,https://quotes.toscrape.com/
9,10,"“A day without sunshine is like, you know, nig...",Steve Martin,"humor, obvious, simile",https://quotes.toscrape.com/


## 13. Save Raw Dataset

Save the collected data in the project's `data/raw/` directory.

The raw dataset is intentionally kept separate from future cleaned and processed datasets.

### Output File

```text
data/raw/scraped_quotes.csv
```

In [13]:
# ============================================================
# SAVE RAW DATASET
# ============================================================

RAW_OUTPUT_PATH = RAW_DATA_DIR / "scraped_quotes.csv"

raw_df.to_csv(
    RAW_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Raw dataset saved successfully:")
print(RAW_OUTPUT_PATH)

Raw dataset saved successfully:
D:\Data-Science-Project\Data & Resources\Web-Intelligence-NLP\data\raw\scraped_quotes.csv


## 14. Verify Saved Dataset

Reload the saved CSV file to confirm that the dataset was written correctly and can be used by downstream notebooks.

In [14]:
# ============================================================
# VERIFY SAVED DATASET
# ============================================================

saved_df = pd.read_csv(RAW_OUTPUT_PATH)

print(f"Reloaded dataset shape: {saved_df.shape}")
display(saved_df.head())

Reloaded dataset shape: (100, 5)


,quote_id,quote_text,author,tags,source_url
0,1,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world",https://quotes.toscrape.com/
1,2,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices",https://quotes.toscrape.com/
2,3,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles",https://quotes.toscrape.com/
3,4,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor",https://quotes.toscrape.com/
4,5,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational",https://quotes.toscrape.com/


## 📊 Scraping Results

The web-scraping stage has now produced a structured raw dataset containing the textual information collected from the selected public practice source.

### Generated Fields

```text
quote_id
quote_text
author
tags
source_url
```

### Dataset Location

```text
data/raw/scraped_quotes.csv
```

In [15]:
# ============================================================
# FINAL SCRAPING SUMMARY
# ============================================================

print("WEB SCRAPING SUMMARY")
print("=" * 60)
print(f"Total Records     : {len(raw_df):,}")
print(f"Total Columns     : {raw_df.shape[1]}")
print(f"Missing Values    : {raw_df.isna().sum().sum():,}")
print(f"Duplicate Rows    : {raw_df.duplicated().sum():,}")
print(f"Output File       : {RAW_OUTPUT_PATH}")
print("=" * 60)
print("✓ Notebook 01 completed successfully.")

WEB SCRAPING SUMMARY
Total Records     : 100
Total Columns     : 5
Missing Values    : 0
Duplicate Rows    : 0
Output File       : D:\Data-Science-Project\Data & Resources\Web-Intelligence-NLP\data\raw\scraped_quotes.csv
✓ Notebook 01 completed successfully.


## 📝 Key Takeaways

- A public practice web source was selected for structured text collection.
- HTTP requests were used to retrieve web pages.
- BeautifulSoup was used to parse HTML.
- Required textual fields were extracted programmatically.
- Pagination was handled to collect multiple pages.
- The collected records were converted into a Pandas DataFrame.
- Basic data-quality checks were performed.
- The raw dataset was saved for downstream NLP processing.

---

## 🔗 Notebook Navigation

**Current:** `01_Web_Scraping.ipynb`

**Next:** `02_Data_Cleaning_and_EDA.ipynb`

---

### 🚀 Next Stage

The next notebook will focus on **Data Cleaning and Exploratory Data Analysis (EDA)**, where the scraped dataset will be inspected, cleaned, analyzed, and prepared for NLP preprocessing.